# BraTS-GLI: 3D U-Net vs SwinUNETR (GPU)

Companion notebook for [MHKhan50/brats-glioma-segmentation](https://github.com/MHKhan50/brats-glioma-segmentation).
**Add Input → Your Work → "BraTS-GLI Download and Preprocess (CPU)"** (its output), and to continue a run, also the output of an earlier version of this notebook.

Kaggle settings: **GPU T4 x2**, **Internet on**, then *Save Version → Save & Run All*. Choose what to train in the cell below;
finished seeds are found in attached outputs and skipped. A session can run up to 12 hours.

Region-based training (WT, TC, ET; sigmoid, Dice + BCE), random 96³ crops, AMP, AdamW with cosine schedule.
Evaluation on the held-out test patients in the **original image space**: Dice and HD95 per region, voxel calibration (ECE), seed ensembles with patient-level bootstrap intervals.
Research only, not a medical device.

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("brats-glioma-segmentation"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/brats-glioma-segmentation.git"], check=True)
os.chdir("brats-glioma-segmentation")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,dev]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
def bs(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "brats_seg.cli", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

In [ ]:
import glob, shutil, json
PREP = os.path.dirname(sorted(glob.glob("/kaggle/input/**/prep/cases.json", recursive=True))[0])
RUNS, REPORTS = "/kaggle/working/runs", "/kaggle/working/reports"
os.makedirs(RUNS, exist_ok=True)
for f in glob.glob("/kaggle/input/**/runs/*_best.pt", recursive=True) + glob.glob("/kaggle/input/**/runs/*_meta.json", recursive=True):
    if not os.path.exists(os.path.join(RUNS, os.path.basename(f))):
        shutil.copy(f, RUNS); print("restored", os.path.basename(f))
cases = json.load(open(PREP + "/cases.json"))
print(PREP, {s: sum(c["split"] == s for c in cases) for s in ("train", "val", "test")})

## Smoke test (a few steps, two validation cases)

In [ ]:
for m in ("unet", "swinunetr"):
    bs("train", "--data", PREP, "--out", "/tmp/smoke", "--model", m, "--epochs", "1", "--limit-steps", "3", "--max-val-cases", "2")
bs("evaluate", "--data", PREP, "--runs", "/tmp/smoke", "--out", "/tmp/smoke/reports", "--limit", "2")
shutil.rmtree("/tmp/smoke", ignore_errors=True)

## Training: edit this cell for each session

In [ ]:
TRAIN = [("unet", "0,1,2")]          # later sessions: [("swinunetr", "0")], [("swinunetr", "1,2")]
EPOCHS = 60
for model, seeds in TRAIN:
    bs("train", "--data", PREP, "--out", RUNS, "--model", model, "--seeds", seeds, "--epochs", str(EPOCHS),
       "--batch-size", "4", "--val-every", "10")

## Test evaluation in native space and summary

In [ ]:
bs("evaluate", "--data", PREP, "--runs", RUNS, "--out", REPORTS)
bs("analyse", "--reports", REPORTS)
for f in glob.glob(RUNS + "/*_meta.json"):
    shutil.copy(f, REPORTS)
shutil.make_archive("/kaggle/working/brats_reports", "zip", REPORTS)